In [1]:
from kafka import KafkaConsumer
import json
import pandas as pd

consumer = KafkaConsumer(
    "transactions.raw",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    consumer_timeout_ms=10000,   # dừng sau 10s không có message mới
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
)

events = [msg.value for msg in consumer]
print(f"Đã đọc {len(events)} event")

df = pd.DataFrame(events)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["ingested_at"] = pd.to_datetime(df["ingested_at"])

/tmp/ipykernel_15543/4003144784.py:5: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Đã đọc 24234 event


In [2]:
df["minute"] = df["ingested_at"].dt.floor("min")
per_minute = df.groupby("minute").size()
print(per_minute)

minute
2026-07-17 17:35:00    43
2026-07-17 17:36:00    50
2026-07-17 17:37:00    37
2026-07-17 17:38:00    54
2026-07-17 17:39:00    51
                       ..
2026-07-24 17:45:00    45
2026-07-24 17:46:00    48
2026-07-24 17:47:00    40
2026-07-24 17:48:00    51
2026-07-24 17:49:00    23
Length: 76, dtype: int64


In [3]:
late_rate = (df["ingested_at"] > df["timestamp"]).mean()
print(f"Late arrival rate: {late_rate:.3f}")  # kỳ vọng gần 0.12

Late arrival rate: 0.119


In [4]:
dup_rate = df["transaction_id"].duplicated().mean()
print(f"Duplicate rate: {dup_rate:.4f}")  # kỳ vọng gần 0.015

Duplicate rate: 0.0146


In [5]:
# balance vẫn phải hợp lý dù sinh liên tục
sample = df.sample(20)
print(sample[["type", "old_balance", "amount", "new_balance", "status"]])

           type  old_balance      amount  new_balance   status
7536    payment   7763710.16   582874.98   7180835.18  success
15914   payment   8519508.23  1467802.77   7051705.46  success
7548    payment  10443417.54  1153557.52   9289860.02  success
2250    deposit  17797029.59    39006.99  17836036.58  success
19607  transfer  24545559.43    19674.88  24525884.55  success
4385    deposit   3697760.87  1191584.64   4889345.51  success
8366    deposit  21646890.03   920975.28  22567865.31  success
10027  transfer   5531227.66  1229102.62   4302125.04  success
1600   transfer   3602237.75   731119.70   2871118.05  success
22224   payment    956430.91   842272.89    114158.02  success
7420   transfer     94834.73  1643544.81     94834.73   failed
22704   payment   2867467.83   560672.82   2306795.01  success
19841  transfer  27943604.86   293451.22  27650153.64  success
2414    payment  15648362.21   778329.75  14870032.46  success
2354    deposit  21818635.41   224940.58  22043575.99  

In [6]:
mask = (
    (df["minute"].dt.date == pd.Timestamp("2026-07-24").date()) &
    (df["minute"].dt.strftime("%H:%M") >= "17:15") &
    (df["minute"].dt.strftime("%H:%M") <= "17:45")
)
print(per_minute[per_minute.index.isin(df.loc[mask, "minute"])])

minute
2026-07-24 17:15:00      54
2026-07-24 17:16:00      66
2026-07-24 17:17:00      55
2026-07-24 17:18:00      56
2026-07-24 17:19:00      61
2026-07-24 17:20:00      56
2026-07-24 17:21:00      42
2026-07-24 17:22:00      54
2026-07-24 17:23:00      64
2026-07-24 17:24:00      39
2026-07-24 17:25:00     796
2026-07-24 17:26:00    1435
2026-07-24 17:27:00    1470
2026-07-24 17:28:00    1474
2026-07-24 17:29:00    1404
2026-07-24 17:30:00    1424
2026-07-24 17:31:00    1441
2026-07-24 17:32:00    1486
2026-07-24 17:33:00    1510
2026-07-24 17:34:00    1511
2026-07-24 17:35:00    1497
2026-07-24 17:36:00    1429
2026-07-24 17:37:00    1461
2026-07-24 17:38:00    1417
2026-07-24 17:39:00    1510
2026-07-24 17:40:00      51
2026-07-24 17:41:00      51
2026-07-24 17:42:00      53
2026-07-24 17:43:00      42
2026-07-24 17:44:00      59
2026-07-24 17:45:00      45
dtype: int64


In [7]:
pd.set_option("display.max_rows", 100)
print(per_minute)

minute
2026-07-17 17:35:00      43
2026-07-17 17:36:00      50
2026-07-17 17:37:00      37
2026-07-17 17:38:00      54
2026-07-17 17:39:00      51
2026-07-17 17:40:00      44
2026-07-17 17:41:00      47
2026-07-17 17:42:00      56
2026-07-17 17:43:00      51
2026-07-17 17:44:00      44
2026-07-17 17:45:00      60
2026-07-17 17:46:00      40
2026-07-17 17:47:00      52
2026-07-17 17:48:00      53
2026-07-17 17:49:00      37
2026-07-17 17:50:00      46
2026-07-17 17:51:00      49
2026-07-17 17:52:00      54
2026-07-17 17:53:00      43
2026-07-17 17:54:00      50
2026-07-17 17:55:00      56
2026-07-17 17:56:00      52
2026-07-17 17:57:00      53
2026-07-17 17:58:00      43
2026-07-17 17:59:00      43
2026-07-17 18:00:00      38
2026-07-17 18:01:00      68
2026-07-17 18:02:00      62
2026-07-17 18:03:00      61
2026-07-17 18:04:00      55
2026-07-17 18:05:00      30
2026-07-17 18:09:00      39
2026-07-17 18:10:00      44
2026-07-17 18:11:00      59
2026-07-17 18:12:00      48
2026-07-17 18